In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
import os
import json
import numpy as np
import pandas as pd
import geopandas as gpd
import textwrap

BASE           = '/content/drive/MyDrive/02_NandiSeedRecommender2'
OUT_DIR        = os.path.join(BASE, 'WardAggregatedData')
FINAL_DIR      = os.path.join(BASE, 'Final_Outputs')
SEED_XLSX_PATH = os.path.join(BASE, 'Seed_Data/KenyaSeedWebScrape.xlsx')

# ---- Load data ----
wards_gdf = gpd.read_file(os.path.join(OUT_DIR, 'Nandi_Ward_Aggregation.geojson'))
county    = gpd.read_file(os.path.join(BASE, 'NandiCounties/counties.shp'))
nandi     = county[county['COUNTY_NAM'] == 'NANDI']

try:
    nandi_union = nandi.union_all()
except AttributeError:
    nandi_union = nandi.geometry.unary_union

wards_gdf = wards_gdf[wards_gdf.geometry.centroid.within(nandi_union)].copy()
name_col  = next((c for c in wards_gdf.columns if 'NAME' in c.upper() or 'WARD' in c.upper()), None)
if name_col:
    wards_gdf = wards_gdf[~wards_gdf[name_col].str.upper().isin(['MANDA-SHIVANGA'])]

if os.path.exists(SEED_XLSX_PATH):
    seed_df = pd.read_excel(SEED_XLSX_PATH, engine='openpyxl')
    seed_df.columns = seed_df.columns.str.strip()
else:
    seed_df = pd.DataFrame()

json_path = os.path.join(FINAL_DIR, 'County_Averages.json')
if os.path.exists(json_path):
    with open(json_path, 'r') as f:
        county_ref_full = json.load(f)
else:
    county_ref_full = {}

# ============================================================
# HELPERS
# ============================================================

RAW_KEY_MAP = {'rain': 'mean_season_rain', 'temp': 'mean_temp', 'rh': 'rh_dev'}

def get_suitability_label(suit_score):
    if np.isnan(suit_score):  return "Unknown"
    if suit_score > 0.75:     return "Very Suitable"
    elif suit_score > 0.50:   return "Moderately Suitable"
    elif suit_score > 0.25:   return "Marginally Suitable"
    else:                     return "Not Suitable"

# ============================================================
# SEED SCORING
# ============================================================

def score_seeds_for_ward(seed_df, elevation, precip, max_temp, min_temp,
                         drought_prob, flood_prob, county_avg_drought, county_avg_flood):
    if seed_df.empty:
        return pd.DataFrame()

    eligible = seed_df[
        (seed_df['Elevation Min'] <= elevation) &
        (seed_df['Elevation Max'] >= elevation)
    ].copy()

    if eligible.empty:
        return pd.DataFrame()

    results = []
    for _, row in eligible.iterrows():
        score      = 0.0
        total_poss = 0.0

        # Precipitation (weight 0.47)
        total_poss += 0.47
        try:
            if row['Precipitation Min'] <= precip <= row['Precipitation Max']:
                score += 0.47
        except:
            pass

        # Elevation fit (weight 0.13)
        total_poss += 0.13
        try:
            elev_min   = row['Elevation Min']
            elev_max   = row['Elevation Max']
            midpoint   = (elev_min + elev_max) / 2
            half_range = (elev_max - elev_min) / 2
            if half_range > 0:
                centrality = 1 - (abs(elevation - midpoint) / half_range)
                score += 0.13 * max(centrality, 0)
            else:
                score += 0.13
        except:
            pass

        # Flood resistance (weight 0.47, only if local > county avg)
        if not np.isnan(flood_prob) and not np.isnan(county_avg_flood):
            if flood_prob > county_avg_flood:
                total_poss += 0.47
                if row.get('Flood-Resistant', 0) == 1:
                    score += 0.47

        # Drought tolerance (weight 0.13, only if local > county avg)
        if not np.isnan(drought_prob) and not np.isnan(county_avg_drought):
            if drought_prob > county_avg_drought:
                total_poss += 0.13
                if row.get('Moisture-Stress Tolerant', 0) == 1:
                    score += 0.13

        # Heat tolerance (weight 0.13, only if max_temp > 35)
        if not np.isnan(max_temp) and max_temp > 35:
            total_poss += 0.13
            if row.get('Heat Tolerant', 0) == 1:
                score += 0.13

        # Cold tolerance (weight 0.13, only if min_temp < 10)
        if not np.isnan(min_temp) and min_temp < 10:
            total_poss += 0.13
            if row.get('Cold Tolerant', 0) == 1:
                score += 0.13

        weighted_score = (score / total_poss) if total_poss > 0 else 0.0

        results.append({
            'Variety':                row['Variety'],
            'Score':                  round(weighted_score * 100, 1),
            'Time to Maturity':       row.get('Time to Maturity', 'N/A'),
            'Potential yield (t/Ha)': row.get('Potential yield (t/Ha)', 'N/A'),
            'Key Attributes':         row.get('Key Attributes', 'N/A'),
        })

    out = pd.DataFrame(results)
    out = out[out['Score'] > 0]

    if out.empty:
        return pd.DataFrame()

    out['YieldNumeric'] = pd.to_numeric(out['Potential yield (t/Ha)'], errors='coerce')
    out = out.sort_values(['Score', 'YieldNumeric'], ascending=[False, False])
    return out.drop(columns='YieldNumeric').head(3)

# ============================================================
# FERTILISER TEXT LOGIC
# ============================================================

def get_fertiliser_text(ph, total_nitrogen, phosphorus_ppm, zinc_score, potassium_cmol):
    lines = []

    if ph < 5.5:
        if total_nitrogen < 0.2:
            if phosphorus_ppm < 15:
                lines.append(
                    "The soil in your area is acidic and low in phosphorus and nitrogen. "
                    "For optimal maize growth, it is recommended to apply MEA Mazao (250 kg/ha) "
                    "followed by Manvuno top-dressing (185 kg/ha) to increase nitrogen and phosphorus "
                    "levels while reducing acidity. Alternatively, apply 100 kg/ha of DAP followed by "
                    "81 kg/ha of CAN approximately six weeks after planting. "
                    "Please note that if you are intercropping cereal and beans, you should increase "
                    "DAP by 17 kg/ac; if you are intercropping cereal and legumes, increase DAP by 27 kg/ac."
                )
            else:
                lines.append(
                    "The soil in your area is acidic and low in nitrogen, but it is rich in phosphorus. "
                    "No phosphorus fertiliser is needed. For optimal maize growth, it is recommended to "
                    "apply 150 kg/ha of CAN to increase nitrogen concentration and reduce acidity."
                )
        else:
            if phosphorus_ppm > 15:
                lines.append(
                    "The soil in your area is high in phosphorus and nitrogen, but it is acidic. "
                    "For optimal maize growth, liming is recommended to reduce acidity."
                )
            else:
                lines.append(
                    "The soil in your area is high in nitrogen, but it is acidic and low in phosphorus. "
                    "For optimal maize growth, it is recommended to apply 100 kg/ha of TSP at planting "
                    "to increase phosphorus concentration without increasing soil acidity. "
                    "Liming may also be beneficial. "
                    "Please note that if you are intercropping cereal and beans, you should increase "
                    "TSP by 17 kg/ac; if you are intercropping cereal and legumes, increase TSP by 27 kg/ac."
                )
    else:
        if total_nitrogen < 0.2:
            if phosphorus_ppm < 15:
                lines.append(
                    "The soil in your area is low in phosphorus and nitrogen, but the acidity levels "
                    "are good. For optimal maize growth, it is recommended to apply 100 kg/ha of DAP "
                    "at planting. Side-dress with the most cost-effective nitrogen source, 46 kg/ha of "
                    "Urea, six weeks after planting. This will increase both phosphorus and nitrogen "
                    "concentrations. "
                    "Please note that if you are intercropping cereal and beans, you should increase "
                    "DAP by 17 kg/ac; if you are intercropping cereal and legumes, increase DAP by "
                    "27 kg/ac and reduce urea by 22 kg/ac."
                )
            else:
                lines.append(
                    "The soil in your area is low in nitrogen, but the phosphorus levels and acidity "
                    "are good. For optimal maize growth, side-dressing 85 kg/ha of Urea is recommended "
                    "to meet the nitrogen requirement. No phosphorus or liming is required. "
                    "Please note that if you are intercropping cereal and legumes, it is best to reduce "
                    "urea by 22 kg/ac."
                )
        else:
            if phosphorus_ppm > 15:
                lines.append(
                    "The soil in your area is high in phosphorus and nitrogen, and has optimal acidity "
                    "for maize growth. No nitrogen or phosphorus-based fertilisers are required, and "
                    "liming is not necessary."
                )
            else:
                lines.append(
                    "The soil in your area is high in nitrogen and has good acidity for maize growth, "
                    "but it is low in phosphorus. For optimal maize growth, it is recommended to apply "
                    "100 kg/ha of TSP at planting to increase phosphorus concentration without increasing "
                    "soil acidity. Liming may also be beneficial. "
                    "Please note that if you are intercropping cereal and beans, you should increase "
                    "TSP by 17 kg/ac; if you are intercropping cereal and legumes, increase TSP by 27 kg/ac."
                )

    if not np.isnan(zinc_score) and zinc_score < 1.0:
        lines.append(
            "Your soil may be deficient in zinc and could benefit from zinc sulfate fertiliser. "
            "A zinc soil test is recommended."
        )

    if not np.isnan(potassium_cmol) and potassium_cmol < 0.256:
        lines.append(
            "Your soil is deficient in potassium. For optimal maize growth, band application of "
            "50 kg/ha of potassium chloride is recommended."
        )

    lines.append(
        "If a field extension officer is available, we recommend consulting them to refine "
        "these guidelines for your specific plot."
    )

    return lines

# ============================================================
# CORE DATA EXTRACTION (shared by print, CSV, and SMS)
# ============================================================

def extract_season_data(ward, season, county_ref):
    pfx_fact     = 'LR_Fact' if 'Long' in season else 'SR_Fact'
    pfx_raw      = 'LR_Raw'  if 'Long' in season else 'SR_Raw'
    pfx_main     = 'Main'
    season_label = 'Long Rains' if 'Long' in season else 'Short Rains'
    planting     = "March 25 to April 20" if 'Long' in season else "October 25 to November 15"

    def raw(var):
        rk = RAW_KEY_MAP.get(var, var)
        return ward.get(f'{pfx_raw}_{rk}_raw', np.nan)

    suit_mu    = ward.get(f'{pfx_main}_Suit_Mean_{season}', np.nan)
    suit_label = get_suitability_label(suit_mu)
    suit_pct   = f"{suit_mu*100:.1f}" if not np.isnan(suit_mu) else "N/A"

    elev_col   = next((c for c in ward.index if 'elev' in c.lower()), None)
    elev_val   = ward[elev_col] if elev_col else np.nan
    precip_val = raw('rain')
    mean_temp  = ward.get(f'{pfx_raw}_mean_temp_raw', np.nan)
    max_temp   = ward.get(f'{pfx_raw}_max_temp_raw',  np.nan)
    min_temp   = ward.get(f'{pfx_raw}_min_temp_raw',  np.nan)

    drought_prob       = ward.get(f'{pfx_fact}_prob_drought_mean', np.nan)
    flood_prob         = ward.get(f'{pfx_fact}_prob_flood_mean',   np.nan)
    county_avg_drought = county_ref.get('scores', {}).get('prob_drought', np.nan)
    county_avg_flood   = county_ref.get('scores', {}).get('prob_flood',   np.nan)

    ph_val   = raw('ph')
    tn_val   = raw('total_nitrogen')
    p_val    = raw('phosphorus')
    k_val    = raw('potassium')
    zn_score = ward.get(f'{pfx_fact}_zinc_mean', np.nan)

    top_seeds = score_seeds_for_ward(
        seed_df,
        elevation=elev_val   if not np.isnan(elev_val)   else 1500,
        precip=precip_val    if not np.isnan(precip_val) else 600,
        max_temp=max_temp,   min_temp=min_temp,
        drought_prob=drought_prob,         flood_prob=flood_prob,
        county_avg_drought=county_avg_drought, county_avg_flood=county_avg_flood,
    )

    fert_lines = None
    if not any(np.isnan(v) for v in [ph_val, tn_val, p_val]):
        fert_lines = get_fertiliser_text(ph_val, tn_val, p_val, zn_score, k_val)

    return {
        'season_label':  season_label,
        'planting':      planting,
        'suit_mu':       suit_mu,
        'suit_pct':      suit_pct,
        'suit_label':    suit_label,
        'elev_val':      elev_val,
        'precip_val':    precip_val,
        'mean_temp':     mean_temp,
        'max_temp':      max_temp,
        'min_temp':      min_temp,
        'ph_val':        ph_val,
        'tn_val':        tn_val,
        'p_val':         p_val,
        'k_val':         k_val,
        'zn_score':      zn_score,
        'drought_prob':  drought_prob,
        'flood_prob':    flood_prob,
        'top_seeds':     top_seeds,
        'fert_lines':    fert_lines,
    }

# ============================================================
# PRINT REPORT
# ============================================================

def print_season_block(d, W):
    sl = d['season_label']

    elev_str  = f"{d['elev_val']:.0f} m"    if not np.isnan(d['elev_val'])   else "N/A"
    rain_str  = f"{d['precip_val']:.0f} mm"  if not np.isnan(d['precip_val']) else "N/A"
    meant_str = f"{d['mean_temp']:.1f} °C"   if not np.isnan(d['mean_temp'])  else "N/A"
    maxt_str  = f"{d['max_temp']:.1f} °C"    if not np.isnan(d['max_temp'])   else "N/A"
    ph_str    = f"{d['ph_val']:.2f}"          if not np.isnan(d['ph_val'])     else "N/A"
    tn_str    = f"{d['tn_val']:.3f} %"        if not np.isnan(d['tn_val'])     else "N/A"
    p_str     = f"{d['p_val']:.1f} mg/kg"     if not np.isnan(d['p_val'])      else "N/A"

    print(f"\n{'─'*W}")
    print(f" FOR THE {sl.upper()} SEASON:")
    print(f"{'─'*W}")
    print(f"\nThe conditions in your area are {d['suit_pct']}% ({d['suit_label']}) suitable for "
          f"maize growth, based on the local soil, weather conditions and terrain.")
    print(f"Elevation: {elev_str}  |  Rainfall: {rain_str}  |  Mean Temperature: {meant_str}  |  "
          f"Max Temperature: {maxt_str}  |  pH: {ph_str}  |  Nitrogen: {tn_str}  |  "
          f"Phosphorus: {p_str}\n")

    print("Below are the top 3 recommended seed varieties based on your location.\n")
    if not d['top_seeds'].empty:
        print(f"  {'Variety':<14} {'Score':>7}%  {'Time to Maturity':<22} "
              f"{'Yield (t/Ha)':<14} Key Attributes")
        print(f"  {'─'*105}")
        for _, r in d['top_seeds'].iterrows():
            print(f"  {r['Variety']:<14} {r['Score']:>7.1f}%  "
                  f"{str(r['Time to Maturity']):<22} "
                  f"{str(r['Potential yield (t/Ha)']):<14} "
                  f"{str(r['Key Attributes'])}")
    else:
        print("  No seed varieties matched the elevation criteria for this ward.")

    print(f"\nFor the {sl} season, the optimal planting time for maize in your "
          f"region is {d['planting']}.\n")

    print("Below are fertiliser recommendations based on the soil properties in your location. "
          "Nitrogen-based fertiliser can improve yields by an average of 2.634 t/ha in your region, "
          "while phosphorus-based fertilisers improve yields by 0.640 t/ha.\n")

    if d['fert_lines'] is None:
        print("  Insufficient soil data to generate fertiliser recommendation for this ward.\n")
    else:
        for line in d['fert_lines']:
            wrapped = textwrap.fill(line, width=W - 4, subsequent_indent='  ')
            print(f"  {wrapped}\n")

    if not np.isnan(d['drought_prob']) and d['drought_prob'] > 0.15:
        print(f"⚠  Warning: During the {sl} season, your area has a historically high "
              f"drought risk ({d['drought_prob']*100:.1f}% of seasons). Consider staged fertiliser "
              f"application and prioritise drought-tolerant seed varieties.\n")

    if not np.isnan(d['flood_prob']) and d['flood_prob'] > 0.15:
        print(f"⚠  Warning: During the {sl} season, your area has a historically high "
              f"flood risk ({d['flood_prob']*100:.1f}% of seasons). Avoid basal fertiliser "
              f"application immediately before forecast heavy rainfall.\n")


def get_ward_report(ward_name):
    ward_row = wards_gdf[wards_gdf[name_col].str.upper() == ward_name.upper()]
    if ward_row.empty:
        print(f"Ward '{ward_name}' not found.")
        print("Available:", sorted(wards_gdf[name_col].tolist()))
        return
    ward = ward_row.iloc[0]
    W    = 115

    print(f"\n{'='*W}")
    print(f"  NANDI PRECISION AGRICULTURE REPORT  |  {ward_name.upper()}")
    print(f"{'='*W}")

    for season in ['LongRains', 'ShortRains']:
        county_ref = county_ref_full.get(season, {'scores': {}, 'raw': {}})
        d = extract_season_data(ward, season, county_ref)
        print_season_block(d, W)

    print(f"\n{'='*W}\n")

# ============================================================
# CSV EXPORT (full text, one row per ward)
# ============================================================

def get_ward_csv_row(ward_name):
    ward_row = wards_gdf[wards_gdf[name_col].str.upper() == ward_name.upper()]
    if ward_row.empty:
        return None
    ward     = ward_row.iloc[0]
    row_data = {'Ward': ward_name}

    for season in ['LongRains', 'ShortRains']:
        county_ref = county_ref_full.get(season, {'scores': {}, 'raw': {}})
        d  = extract_season_data(ward, season, county_ref)
        s  = 'LR' if 'Long' in season else 'SR'
        sl = d['season_label']

        elev_str  = f"{d['elev_val']:.0f} m"    if not np.isnan(d['elev_val'])   else "N/A"
        rain_str  = f"{d['precip_val']:.0f} mm"  if not np.isnan(d['precip_val']) else "N/A"
        meant_str = f"{d['mean_temp']:.1f} C"    if not np.isnan(d['mean_temp'])  else "N/A"
        maxt_str  = f"{d['max_temp']:.1f} C"     if not np.isnan(d['max_temp'])   else "N/A"
        ph_str    = f"{d['ph_val']:.2f}"          if not np.isnan(d['ph_val'])     else "N/A"
        tn_str    = f"{d['tn_val']:.3f}%"         if not np.isnan(d['tn_val'])     else "N/A"
        p_str     = f"{d['p_val']:.1f} mg/kg"     if not np.isnan(d['p_val'])      else "N/A"

        row_data[f'{s}_Suitability'] = (
            f"The conditions in your area are {d['suit_pct']}% ({d['suit_label']}) suitable for maize growth, "
            f"based on the local soil, weather conditions and terrain. "
            f"Elevation: {elev_str} | Rainfall: {rain_str} | Mean Temp: {meant_str} | "
            f"Max Temp: {maxt_str} | pH: {ph_str} | Nitrogen: {tn_str} | Phosphorus: {p_str}"
        )

        if not d['top_seeds'].empty:
            seed_lines = []
            for _, r in d['top_seeds'].iterrows():
                seed_lines.append(
                    f"{r['Variety']} (Score: {r['Score']}% | Yield: {r['Potential yield (t/Ha)']} t/Ha | "
                    f"Maturity: {r['Time to Maturity']} | {r['Key Attributes']})"
                )
            row_data[f'{s}_Seeds'] = "Top recommended seed varieties: " + " | ".join(seed_lines)
        else:
            row_data[f'{s}_Seeds'] = "No seed varieties matched the elevation criteria for this ward."

        row_data[f'{s}_Planting_Window'] = (
            f"For the {sl} season, the optimal planting time for maize in your region is {d['planting']}."
        )

        if d['fert_lines'] is None:
            row_data[f'{s}_Fertiliser'] = "Insufficient soil data to generate fertiliser recommendation."
        else:
            fert_intro = (
                "Nitrogen-based fertiliser can improve yields by an average of 2.634 t/ha in your region, "
                "while phosphorus-based fertilisers improve yields by 0.640 t/ha. "
            )
            row_data[f'{s}_Fertiliser'] = fert_intro + " ".join(d['fert_lines'])

        warnings = []
        if not np.isnan(d['drought_prob']) and d['drought_prob'] > 0.15:
            warnings.append(
                f"WARNING: During the {sl} season, your area has a historically high drought risk "
                f"({d['drought_prob']*100:.1f}% of seasons). Consider staged fertiliser application "
                f"and prioritise drought-tolerant seed varieties."
            )
        if not np.isnan(d['flood_prob']) and d['flood_prob'] > 0.15:
            warnings.append(
                f"WARNING: During the {sl} season, your area has a historically high flood risk "
                f"({d['flood_prob']*100:.1f}% of seasons). Avoid basal fertiliser application "
                f"immediately before forecast heavy rainfall."
            )
        row_data[f'{s}_Risk_Warnings'] = " | ".join(warnings) if warnings else ""

    return row_data


def export_all_wards_csv(out_path=None):
    if out_path is None:
        out_path = os.path.join(OUT_DIR, 'Nandi_Ward_Recommendations.csv')

    ward_names = sorted(wards_gdf[name_col].tolist())
    records    = []
    for ward_name in ward_names:
        print(f"  Processing {ward_name}...")
        data = get_ward_csv_row(ward_name)
        if data:
            records.append(data)

    df = pd.DataFrame(records)
    df.to_csv(out_path, index=False)
    print(f"\nSaved {len(records)} wards to {out_path}")
    return df

# ============================================================
# SMS EXPORT (segmented messages, one row per ward)
# ============================================================

def build_sms_messages(ward_name):
    ward_row = wards_gdf[wards_gdf[name_col].str.upper() == ward_name.upper()]
    if ward_row.empty:
        return {}
    ward   = ward_row.iloc[0]
    output = {}

    for season in ['LongRains', 'ShortRains']:
        county_ref = county_ref_full.get(season, {'scores': {}, 'raw': {}})
        d        = extract_season_data(ward, season, county_ref)
        sl       = d['season_label'].upper()
        messages = []

        # MSG 1: Suitability + key stats
        elev_str  = f"{d['elev_val']:.0f}m"    if not np.isnan(d['elev_val'])   else "N/A"
        rain_str  = f"{d['precip_val']:.0f}mm"  if not np.isnan(d['precip_val']) else "N/A"
        meant_str = f"{d['mean_temp']:.1f}C"    if not np.isnan(d['mean_temp'])  else "N/A"
        maxt_str  = f"{d['max_temp']:.1f}C"     if not np.isnan(d['max_temp'])   else "N/A"
        ph_str    = f"{d['ph_val']:.2f}"         if not np.isnan(d['ph_val'])     else "N/A"
        tn_str    = f"{d['tn_val']:.3f}%"        if not np.isnan(d['tn_val'])     else "N/A"
        p_str     = f"{d['p_val']:.1f}mg/kg"     if not np.isnan(d['p_val'])      else "N/A"

        messages.append(
            f"{ward_name.upper()} - {sl}\n"
            f"Suitability: {d['suit_pct']}% ({d['suit_label']})\n"
            f"Elev: {elev_str} | Rain: {rain_str} | Temp: {meant_str} max {maxt_str}\n"
            f"pH: {ph_str} | N: {tn_str} | P: {p_str}"
        )

        # MSG 2: Seeds
        if not d['top_seeds'].empty:
            seed_lines = []
            for i, (_, r) in enumerate(d['top_seeds'].iterrows(), 1):
                seed_lines.append(
                    f"{i}. {r['Variety']} ({r['Score']}%) - "
                    f"{r['Potential yield (t/Ha)']}t/ha, {r['Time to Maturity']}. "
                    f"{r['Key Attributes']}"
                )
            messages.append("SEEDS\n" + "\n".join(seed_lines))
        else:
            messages.append("SEEDS\nNo varieties matched your elevation.")

        # MSG 3: Planting window
        messages.append(f"PLANTING\n{d['season_label']}: plant {d['planting']}")

        # MSG 4: Fertiliser
        if d['fert_lines'] is None:
            messages.append("FERTILISER\nInsufficient soil data.")
        else:
            fert_intro = "Nitrogen fertiliser improves yield by 2.6t/ha in your region, and phosphorus by 0.6t/ha. "
            fert_body  = " ".join(d['fert_lines'])
            replacements = [
                ("For optimal maize growth, it is recommended to apply", "Apply"),
                ("For optimal maize growth, it is recommended to",       ""),
                ("it is recommended to",                                  ""),
                ("approximately six weeks after planting",               "6 weeks after planting"),
                ("Please note that if you are intercropping",            "Intercrop"),
                ("you should increase",                                   "increase"),
                ("If a field extension officer is available, we recommend consulting them to refine these guidelines for your specific plot.",
                 "Consult extension officer to refine."),
                ("could benefit from zinc sulfate fertiliser. A zinc soil test is recommended.",
                 "Zinc sulfate may help. Soil test recommended."),
                ("band application of",  "band-apply"),
                ("it is best to reduce", "reduce"),
            ]
            for old, new in replacements:
                fert_body = fert_body.replace(old, new)
            messages.append(f"FERTILISER\n{fert_intro}{fert_body.strip()}")

        # MSG 5: Risk warnings (only if triggered)
        warnings = []
        if not np.isnan(d['drought_prob']) and d['drought_prob'] > 0.15:
            warnings.append(
                f"DROUGHT RISK: {d['drought_prob']*100:.1f}% of seasons. "
                f"Use staged fertiliser. Prioritise drought-tolerant seeds."
            )
        if not np.isnan(d['flood_prob']) and d['flood_prob'] > 0.15:
            warnings.append(
                f"FLOOD RISK: {d['flood_prob']*100:.1f}% of seasons. "
                f"Avoid basal fertiliser before heavy rain."
            )
        if warnings:
            messages.append("WARNINGS\n" + "\n".join(warnings))

        total = len(messages)
        output[season] = [
            f"MSG {i+1}/{total} - {m}" for i, m in enumerate(messages)
        ]

    return output


def print_sms_preview(ward_name):
    sms = build_sms_messages(ward_name)
    for season, msgs in sms.items():
        season_label = 'Long Rains' if 'Long' in season else 'Short Rains'
        print(f"\n{'='*60}")
        print(f"  {season_label.upper()}")
        print(f"{'='*60}")
        for msg in msgs:
            print(f"\n┌{'─'*58}┐")
            for line in msg.split('\n'):
                for chunk in textwrap.wrap(line, width=56) or ['']:
                    print(f"│ {chunk:<56} │")
            print(f"└{'─'*58}┘")
            print(f"  {len(msg)} chars")


def export_sms_csv(out_path=None):
    if out_path is None:
        out_path = os.path.join(OUT_DIR, 'Nandi_Ward_SMS.csv')

    ward_names = sorted(wards_gdf[name_col].tolist())
    records    = []

    for ward_name in ward_names:
        print(f"  Processing {ward_name}...")
        sms = build_sms_messages(ward_name)
        row = {'Ward': ward_name}
        for season, msgs in sms.items():
            pfx = 'LR' if 'Long' in season else 'SR'
            for i, msg in enumerate(msgs, 1):
                row[f'{pfx}_MSG{i}'] = msg
        records.append(row)

    df = pd.DataFrame(records)
    df.to_csv(out_path, index=False)
    print(f"\nSaved {len(records)} wards to {out_path}")
    return df

# ============================================================
# RUN
# ============================================================

# Full formatted print report for one ward:
get_ward_report('CHEMELIL/CHEMASE')

# SMS preview for one ward:
print_sms_preview('CHEMELIL/CHEMASE')

# Export full text CSV for all wards:
export_all_wards_csv()

# Export SMS-optimised CSV for all wards:
# export_sms_csv()

/tmp/ipython-input-2812255699.py:23: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  wards_gdf = wards_gdf[wards_gdf.geometry.centroid.within(nandi_union)].copy()



  NANDI PRECISION AGRICULTURE REPORT  |  CHEMELIL/CHEMASE

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────
 FOR THE LONG RAINS SEASON:
───────────────────────────────────────────────────────────────────────────────────────────────────────────────────

The conditions in your area are 53.3% (Moderately Suitable) suitable for maize growth, based on the local soil, weather conditions and terrain.
Elevation: 1439 m  |  Rainfall: 705 mm  |  Mean Temperature: 20.1 °C  |  Max Temperature: 25.3 °C  |  pH: 5.89  |  Nitrogen: 0.165 %  |  Phosphorus: 12.2 mg/kg

Below are the top 3 recommended seed varieties based on your location.

  Variety          Score%  Time to Maturity       Yield (t/Ha)   Key Attributes
  ─────────────────────────────────────────────────────────────────────────────────────────────────────────
  H513              80.5%  110 - 130 Days         5.72           Sweet tasting; good standability; partially toler

,Ward,LR_Suitability,LR_Seeds,LR_Planting_Window,LR_Fertiliser,LR_Risk_Warnings,SR_Suitability,SR_Seeds,SR_Planting_Window,SR_Fertiliser,SR_Risk_Warnings
0,CHEMELIL/CHEMASE,The conditions in your area are 53.3% (Moderat...,Top recommended seed varieties: H513 (Score: 8...,"For the Long Rains season, the optimal plantin...",Nitrogen-based fertiliser can improve yields b...,,The conditions in your area are 53.8% (Moderat...,Top recommended seed varieties: DH04 (Score: 9...,"For the Short Rains season, the optimal planti...",Nitrogen-based fertiliser can improve yields b...,"WARNING: During the Short Rains season, your a..."
1,CHEMUNDU/KAPNG'ETUNY,The conditions in your area are 62.8% (Moderat...,Top recommended seed varieties: H9401 (Score: ...,"For the Long Rains season, the optimal plantin...",Nitrogen-based fertiliser can improve yields b...,"WARNING: During the Long Rains season, your ar...",The conditions in your area are 58.3% (Moderat...,Top recommended seed varieties: H6210 (Score: ...,"For the Short Rains season, the optimal planti...",Nitrogen-based fertiliser can improve yields b...,"WARNING: During the Short Rains season, your a..."
2,CHEPKUMIA,The conditions in your area are 59.5% (Moderat...,Top recommended seed varieties: H9401 (Score: ...,"For the Long Rains season, the optimal plantin...",Nitrogen-based fertiliser can improve yields b...,"WARNING: During the Long Rains season, your ar...",The conditions in your area are 59.5% (Moderat...,Top recommended seed varieties: H9401 (Score: ...,"For the Short Rains season, the optimal planti...",Nitrogen-based fertiliser can improve yields b...,"WARNING: During the Short Rains season, your a..."
3,CHEPKUNYUK,The conditions in your area are 63.5% (Moderat...,Top recommended seed varieties: H9401 (Score: ...,"For the Long Rains season, the optimal plantin...",Nitrogen-based fertiliser can improve yields b...,"WARNING: During the Long Rains season, your ar...",The conditions in your area are 55.7% (Moderat...,Top recommended seed varieties: H9401 (Score: ...,"For the Short Rains season, the optimal planti...",Nitrogen-based fertiliser can improve yields b...,"WARNING: During the Short Rains season, your a..."
4,CHEPTERWAI,The conditions in your area are 59.6% (Moderat...,Top recommended seed varieties: H9401 (Score: ...,"For the Long Rains season, the optimal plantin...",Nitrogen-based fertiliser can improve yields b...,,The conditions in your area are 59.5% (Moderat...,Top recommended seed varieties: H6210 (Score: ...,"For the Short Rains season, the optimal planti...",Nitrogen-based fertiliser can improve yields b...,"WARNING: During the Short Rains season, your a..."
5,KABISAGA,The conditions in your area are 63.2% (Moderat...,Top recommended seed varieties: H6210 (Score: ...,"For the Long Rains season, the optimal plantin...",Nitrogen-based fertiliser can improve yields b...,"WARNING: During the Long Rains season, your ar...",The conditions in your area are 46.2% (Margina...,Top recommended seed varieties: H6210 (Score: ...,"For the Short Rains season, the optimal planti...",Nitrogen-based fertiliser can improve yields b...,"WARNING: During the Short Rains season, your a..."
6,KABIYET,The conditions in your area are 62.8% (Moderat...,Top recommended seed varieties: H6210 (Score: ...,"For the Long Rains season, the optimal plantin...",Nitrogen-based fertiliser can improve yields b...,"WARNING: During the Long Rains season, your ar...",The conditions in your area are 49.7% (Margina...,Top recommended seed varieties: H6210 (Score: ...,"For the Short Rains season, the optimal planti...",Nitrogen-based fertiliser can improve yields b...,"WARNING: During the Short Rains season, your a..."
7,KABWARENG,The conditions in your area are 59.3% (Moderat...,Top recommended seed varieties: H9401 (Score: ...,"For the Long Rains season, the optimal plantin...",Nitrogen-based fertiliser can improve yields b...,,The conditions in your area are 59.3% (Moderat...,Top recommended seed varie

In [9]:
export_all_wards_csv(out_path='/content/drive/MyDrive/02_NandiSeedRecommender2/JuliaCSV/Nandi_Ward_Recommendations.csv')

  Processing CHEMELIL/CHEMASE...
  Processing CHEMUNDU/KAPNG'ETUNY...
  Processing CHEPKUMIA...
  Processing CHEPKUNYUK...
  Processing CHEPTERWAI...
  Processing KABISAGA...
  Processing KABIYET...
  Processing KABWARENG...
  Processing KAPCHORUA...
  Processing KAPKANGANI...
  Processing KAPSABET...
  Processing KAPSIMOTWO...
  Processing KAPTEL/KAMOIYWO...
  Processing KAPTUMO-KABOI...
  Processing KEMELOI-MARABA...
  Processing KILIBWONI...
  Processing KIPKAREN...
  Processing KIPTUYA...
  Processing KOBUJOI...
  Processing KOSIRAI...
  Processing KOYO-NDURIO...
  Processing KURGUNG/SURUNGAI...
  Processing LELMOKWO/NGECHEK...
  Processing NANDI HILLS...
  Processing NDALAT...
  Processing OL'LESSOS...
  Processing SANGALO/KEBULONIK...
  Processing SONGHOR/SOBA...
  Processing TERIK...
  Processing TINDIRET...

Saved 30 wards to /content/drive/MyDrive/02_NandiSeedRecommender2/JuliaCSV/Nandi_Ward_Recommendations.csv


,Ward,LR_Suitability,LR_Seeds,LR_Planting_Window,LR_Fertiliser,LR_Risk_Warnings,SR_Suitability,SR_Seeds,SR_Planting_Window,SR_Fertiliser,SR_Risk_Warnings
0,CHEMELIL/CHEMASE,The conditions in your area are 53.3% (Moderat...,Top recommended seed varieties: H513 (Score: 8...,"For the Long Rains season, the optimal plantin...",Nitrogen-based fertiliser can improve yields b...,,The conditions in your area are 53.8% (Moderat...,Top recommended seed varieties: DH04 (Score: 9...,"For the Short Rains season, the optimal planti...",Nitrogen-based fertiliser can improve yields b...,"WARNING: During the Short Rains season, your a..."
1,CHEMUNDU/KAPNG'ETUNY,The conditions in your area are 62.8% (Moderat...,Top recommended seed varieties: H9401 (Score: ...,"For the Long Rains season, the optimal plantin...",Nitrogen-based fertiliser can improve yields b...,"WARNING: During the Long Rains season, your ar...",The conditions in your area are 58.3% (Moderat...,Top recommended seed varieties: H6210 (Score: ...,"For the Short Rains season, the optimal planti...",Nitrogen-based fertiliser can improve yields b...,"WARNING: During the Short Rains season, your a..."
2,CHEPKUMIA,The conditions in your area are 59.5% (Moderat...,Top recommended seed varieties: H9401 (Score: ...,"For the Long Rains season, the optimal plantin...",Nitrogen-based fertiliser can improve yields b...,"WARNING: During the Long Rains season, your ar...",The conditions in your area are 59.5% (Moderat...,Top recommended seed varieties: H9401 (Score: ...,"For the Short Rains season, the optimal planti...",Nitrogen-based fertiliser can improve yields b...,"WARNING: During the Short Rains season, your a..."
3,CHEPKUNYUK,The conditions in your area are 63.5% (Moderat...,Top recommended seed varieties: H9401 (Score: ...,"For the Long Rains season, the optimal plantin...",Nitrogen-based fertiliser can improve yields b...,"WARNING: During the Long Rains season, your ar...",The conditions in your area are 55.7% (Moderat...,Top recommended seed varieties: H9401 (Score: ...,"For the Short Rains season, the optimal planti...",Nitrogen-based fertiliser can improve yields b...,"WARNING: During the Short Rains season, your a..."
4,CHEPTERWAI,The conditions in your area are 59.6% (Moderat...,Top recommended seed varieties: H9401 (Score: ...,"For the Long Rains season, the optimal plantin...",Nitrogen-based fertiliser can improve yields b...,,The conditions in your area are 59.5% (Moderat...,Top recommended seed varieties: H6210 (Score: ...,"For the Short Rains season, the optimal planti...",Nitrogen-based fertiliser can improve yields b...,"WARNING: During the Short Rains season, your a..."
5,KABISAGA,The conditions in your area are 63.2% (Moderat...,Top recommended seed varieties: H6210 (Score: ...,"For the Long Rains season, the optimal plantin...",Nitrogen-based fertiliser can improve yields b...,"WARNING: During the Long Rains season, your ar...",The conditions in your area are 46.2% (Margina...,Top recommended seed varieties: H6210 (Score: ...,"For the Short Rains season, the optimal planti...",Nitrogen-based fertiliser can improve yields b...,"WARNING: During the Short Rains season, your a..."
6,KABIYET,The conditions in your area are 62.8% (Moderat...,Top recommended seed varieties: H6210 (Score: ...,"For the Long Rains season, the optimal plantin...",Nitrogen-based fertiliser can improve yields b...,"WARNING: During the Long Rains season, your ar...",The conditions in your area are 49.7% (Margina...,Top recommended seed varieties: H6210 (Score: ...,"For the Short Rains season, the optimal planti...",Nitrogen-based fertiliser can improve yields b...,"WARNING: During the Short Rains season, your a..."
7,KABWARENG,The conditions in your area are 59.3% (Moderat...,Top recommended seed varieties: H9401 (Score: ...,"For the Long Rains season, the optimal plantin...",Nitrogen-based fertiliser can improve yields b...,,The conditions in your area are 59.3% (Moderat...,Top recommended seed varie